# Guide 12 — Free Hints & the Pre-Game Riddle

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## What is this notebook about?

During the pre-game, two kinds of messages arrive:

1. The **riddle** (solved by the AI in Guide 6).
2. A **free hint** that everyone gets — but it arrives in **pieces**, and you have to
   collect all the pieces and put them back in order before you can read it. Like
   getting a sentence one word at a time and reassembling it.

The code below is the real code from the match program that handles these messages.
It's part of the bigger match "client," so it uses `self`.


### How this guide fits in

**Depends on:** Guides 6 (riddle solver) and 11 (number-hint reader). **Used by:** the main turn loop (Guide 0).

*New here? Read **Guide 0 — How Everything Connects** first for the big picture.*


### Solving the riddle in the background

When the riddle arrives, we solve it on a separate "thread" so the game doesn't
freeze while the AI thinks. This function updates the on-screen stage as it goes.


In [ ]:
    def _solve_pregame_riddle_background(self, riddle_text):
        """Solve off the poll thread so slow LLM I/O never stops referee messages."""
        try:
            result = solve_pregame_riddle_with_llm(riddle_text)
            with self.lock:
                if self.pregame_riddle != riddle_text or self._stop_event.is_set():
                    return
                self.pregame_riddle_answer = result['answer']
                self.stage = Stage.RIDDLE_PROCESSED
            self.on_update()
            fallback_note = ' (plain-text fallback)' if result.get('used_fallback') else ''
            print(f'>>> PRE-GAME RIDDLE ANSWER{fallback_note}: {result["answer"]} <<<')
            with self.lock:
                if self.pregame_riddle == riddle_text and not self._stop_event.is_set():
                    self.stage = Stage.RIDDLE_PRINTED
            self.on_update()
        except Exception as exc:
            print(f'[match] could not auto-solve the pre-game riddle: {exc}')

### Handling the messages as they arrive

This is the part of the message-reader that deals with the riddle and the free-hint
pieces. Look at the `free_hint_fragment` part: it saves each piece by its number, and
once **all** the pieces have arrived, it joins them back into one sentence.


In [ ]:
        elif message_type == 'pregame_riddle':
            # Sent to both teams during the pre-game window. Auto-solved via
            # the halo Strix LLM below -- tell the human referee out loud if
            # you're first to answer correctly. There is no wire message for
            # submitting an answer, judging is entirely manual.
            with self.lock:
                self.pregame_riddle = message['riddle']
                self.pregame_riddle_answer = None
                self.free_hint_fragments = {}
                self.free_hint_total = None
                self.free_hint_text = None
                self.stage = Stage.RIDDLE_RECEIVED
            self.on_update()
            threading.Thread(
                target=self._solve_pregame_riddle_background,
                args=(self.pregame_riddle,),
                daemon=True,
                name='pregame-riddle-solver',
            ).start()
        elif message_type == 'free_hint_fragment':
            # One shared, non-competitive hint, split into plain-text
            # fragments -- assemble every index 0..total-1 yourself.
            with self.lock:
                self.free_hint_fragments[message['index']] = message['text']
                self.free_hint_total = message['total']
                self.stage = Stage.FREE_HINT_RECEIVED
                if len(self.free_hint_fragments) == self.free_hint_total:
                    ordered = [self.free_hint_fragments[i] for i in range(self.free_hint_total)]
                    self.free_hint_text = ' '.join(ordered)
                    self.stage = Stage.FREE_HINT_PROCESSED
            if self.stage == Stage.FREE_HINT_PROCESSED:
                self.on_update()
                with self.lock:
                    self.stage = Stage.FREE_HINT_READY

### Check yourself

1. Why solve the riddle on a separate thread instead of right away?
2. The hint pieces might arrive out of order. How does the code make sure the final
   sentence is put back together correctly?
